In [2]:
import os
import requests
import pandas as pd

In [9]:
BASE_URL = "https://ctppdata.transportation.org/api"

session = requests.Session()
# Note, you must obtain a census key and create a system environment variable called CENSUS_KEY that stores the key value
# You can obtain a key here by creating a login: https://ctppdata.transportation.org
# Select "Manage API Keys" from the menu, then "Generate New Key" on the "Your API Keys" page.
API_KEY = os.environ.get("CENSUS-KEY")
print(API_KEY)
session.headers.update({
     "X-API-Key": API_KEY
})
year = 2021
counties = ["033", "035", "053", "061"]  # King, Kitsap, Pierce, Snohomish

xvktbe0p30r9bqfovquj4gls


In [10]:
all_results = []
county_dict = {
    "033": "King",
    "035": "Kitsap",
    "053": "Pierce",
    "061": "Snohomish"
}

In [11]:
for origin in counties:
    for destination in counties:
        base_params = {
            "for": f"county:{origin}",
            "in": "state:53",
            "d-for": f"county:{destination}",
            "d-in": "state:53",
        }

        resp_e = session.get(f"{BASE_URL}/data/{year}", params={**base_params, "get": "B302100_e1"})
        # resp_m = session.get(f"{BASE_URL}/data/{year}", params={**base_params, "get": "B302100_m1"})
        
        data_e = resp_e.json().get(f"data", [])
        # data_m = resp_m.json().get("data", [])

        est = data_e[0]['b302100_e1'] if data_e else 0
        # moe = data_m[0]['b302100_m1'] if data_m else 0

        print(f"{county_dict.get(origin)} -> {county_dict.get(destination)}: {est} estimated workers")
        all_results.append({
            "home_county": county_dict.get(origin),
            "work_county": county_dict.get(destination),
            "total_workers_observed": est
        })

King -> King: 1,128,910 estimated workers
King -> Kitsap: 1,345 estimated workers
King -> Pierce: 28,080 estimated workers
King -> Snohomish: 35,695 estimated workers
Kitsap -> King: 10,600 estimated workers
Kitsap -> Kitsap: 108,165 estimated workers
Kitsap -> Pierce: 7,210 estimated workers
Kitsap -> Snohomish: 815 estimated workers
Pierce -> King: 105,850 estimated workers
Pierce -> Kitsap: 5,450 estimated workers
Pierce -> Pierce: 313,075 estimated workers
Pierce -> Snohomish: 2,155 estimated workers
Snohomish -> King: 127,290 estimated workers
Snohomish -> Kitsap: 400 estimated workers
Snohomish -> Pierce: 1,230 estimated workers
Snohomish -> Snohomish: 276,150 estimated workers


In [12]:
df = pd.DataFrame(all_results)
df.to_csv(r'R:\e2projects_two\SoundCast\Inputs\db_inputs\observed_commute_flows.csv', index=False)

print("Saved to observed_commute_flows.csv")

Saved to observed_commute_flows.csv
